In [1]:
import pandas as pd
import numpy as np
from zipf import zipf_pmf, fit_powerlaw_mle
from visualize import format_plot
from collections import Counter
import matplotlib.pyplot as plt
from jitter import jitter

# Dataset

In [ ]:
folder = "faers/2025"
drug = pd.concat([pd.read_csv(f"{folder}/DRUG25Q{i + 1}.txt", sep="$", encoding="latin1") for i in range(4)])
out = pd.concat([pd.read_csv(f"{folder}/OUTC25Q{i + 1}.txt", sep="$", encoding="latin1") for i in range(4)])

In [3]:
death_cases = out.query("outc_cod == 'DE'")["caseid"].unique()
hospital_cases = out.query("outc_cod == 'HO'")["caseid"].unique()
life_threat_cases = out.query("outc_cod == 'LT'")["caseid"].unique()
other_serious_cases = out.query("outc_cod == 'OT'")["caseid"].unique()


death_drug = drug[drug["caseid"].isin(death_cases)]
hospital_drug = drug[drug["caseid"].isin(hospital_cases)]
life_threat_drug = drug[drug["caseid"].isin(life_threat_cases)]
other_serious_drug = drug[drug["caseid"].isin(other_serious_cases)]

Filter cases where drugs were associated with death ids

In [4]:
counts = death_drug.groupby("caseid").size()
counts = counts.to_numpy().copy()
n = counts.shape[0]

# Fit to Power-Law

In [5]:
k_hat = np.arange(min(counts), max(counts))
counts_jitter = jitter(counts, min(counts))
s_hat, _, _, _ = fit_powerlaw_mle(counts_jitter)
pmf_hat = zipf_pmf(k_hat, s_hat, int(min(counts)))
n_k_hat = (pmf_hat*n).astype(int)
s_hat

np.float64(1.920065389256588)

In [ ]:
count_freq = Counter(counts)

k = np.array(sorted(count_freq.keys()))
n_k = np.array([count_freq[i] for i in k])

fig = plt.figure(figsize=(16, 9))
ax = fig.add_subplot(111)

ax.scatter(np.log(k), np.log(n_k), color="white")
ax.set_xlabel("Log Quantidade de Drogas")
ax.set_ylabel("Log Número de Casos")
format_plot(ax, "white")
plt.savefig("assets/faers/faers_raw_counts.png", transparent=True)
plt.close()

In [ ]:
k_trunc = k[np.log(n_k) > 5]
n_k_trunc = n_k[np.log(n_k) > 5]

fig = plt.figure(figsize=(16, 9))
ax = fig.add_subplot(111)
ax.scatter(np.log(k_trunc), np.log(n_k_trunc), color="white")
ax.set_xlabel("Log Quantidade de Drogas")
ax.set_ylabel("Log Número de Casos")
format_plot(ax, "white")
plt.savefig("assets/faers/faers_truncated_counts.png", transparent=True)
plt.close()

In [ ]:
fig = plt.figure(figsize=(16, 9))
ax = fig.add_subplot(111)

ax.scatter(np.log(k), np.log(n_k), color="white", label="Empírico")
ax.scatter(np.log(k_hat[n_k_hat != 0]), np.log(n_k_hat[n_k_hat != 0]), color="orange", label="Estimado")
ax.set_xlabel("Log Quantidade de Drogas")
ax.set_ylabel("Log Número de Casos")
format_plot(ax, "white")
plt.savefig("assets/faers/faers_counts_estimated.png", transparent=True)
plt.close()

In [9]:
from mc import mc_single
np.random.seed(0)

R = 100
N = 1_000
outcomes = []
for i in range(R):
    hyp_test = mc_single(np.random.choice(counts_jitter, replace=False, size=N))
    outcomes.append(hyp_test["KS"]["greater"])

p_value = np.mean(outcomes)
print(f"n = {N}; p-value = {p_value}")

n = 1000; p-value = 0.04
